In [1]:
import pandas as pd
import numpy as np

ModuleNotFoundError: No module named 'pandas'

In [2]:
import os

In [3]:
path = '../data/raw/'

In [4]:
file_name = 'laptop.csv'

In [5]:
file_path = path+file_name

In [6]:
file_path

'../data/raw/laptop.csv'

In [7]:
df = pd.read_csv(file_path)
df

,Company,Product,TypeName,Inches,ScreenResolution,CPU_Company,CPU_Type,CPU_Frequency (GHz),RAM (GB),Memory,GPU_Company,GPU_Type,OpSys,Weight (kg),Price (Euro)
0,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel,Core i5,2.3,8,128GB SSD,Intel,Iris Plus Graphics 640,macOS,1.37,1339.69
1,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel,Core i5,1.8,8,128GB Flash Storage,Intel,HD Graphics 6000,macOS,1.34,898.94
2,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel,Core i5 7200U,2.5,8,256GB SSD,Intel,HD Graphics 620,No OS,1.86,575.00
3,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel,Core i7,2.7,16,512GB SSD,AMD,Radeon Pro 455,macOS,1.83,2537.45
4,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel,Core i5,3.1,8,256GB SSD,Intel,Iris Plus Graphics 650,macOS,1.37,1803.60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1270,Lenovo,Yoga 500-14ISK,2 in 1 Convertible,14.0,IPS Panel Full HD / Touchscreen 1920x1080,Intel,Core i7 6500U,2.5,4,128GB SSD,Intel,HD Graphics 520,Windows 10,1.80,638.00
1271,Lenovo,Yoga 900-13ISK,2 in 1 Convertible,13.3,IPS Panel Quad HD+ / Touchscreen 3200x1800,Intel,Core i7 6500U,2.5,16,512GB SSD,Intel,HD Graphics 520,Windows 10,1.30,1499.00
1272,Lenovo,IdeaPad 100S-14IBR,Notebook,14.0,1366x768,Intel,Celeron Dual Core N3050,1.6,2,64GB Flash Storage,Intel,HD Graphics,Windows 10,1.50,229.00
1273,HP,15-AC110nv (i7-6500U/6GB/1TB/Radeon,Notebook,15.6,1366x768,Intel,Core i7 6500U,2.5,6,1TB HDD,AMD,Radeon R5 M330,Windows 10,2.19,764.00


In [8]:
df_clean = pd.DataFrame(df['Company'])

In [9]:
copy_paste_rows = ['RAM (GB)', 'CPU_Frequency (GHz)','Weight (kg)', 'Price (Euro)', 'Inches']

In [10]:
for row in copy_paste_rows:
    df_clean[row] = df[row]

In [11]:
set(df['OpSys'])

{'Android',
 'Chrome OS',
 'Linux',
 'Mac OS X',
 'No OS',
 'Windows 10',
 'Windows 10 S',
 'Windows 7',
 'macOS'}

In [12]:
def transform(ops):
    if ops in ['Windows 10', 'Windows 10 S', 'Windows 7', 'No OS']:
        return 'Windows'
    elif ops in ['Mac OS X','macOS']:
        return 'Mac'
    else:
        return ops
    

In [13]:
clean = df_clean

In [14]:
clean['OpSys'] = [transform(ops) for ops in df['OpSys']]

In [17]:
df['Memory'] = df['Memory'].str.replace('Flash Storage', 'SSD')

In [19]:
df[['ssd', 'hdd']] = df['Memory'].str.split('+', expand = True)

In [21]:
memory_clean = df['Memory'].astype(str).str.replace('Flash Storage|eMMC', 'SSD', regex=True)

# 2. Extract values based on text content, NOT string position
# If the row contains 'SSD', extract the number sequence just before the word 'SSD'
df['ssd'] = np.where(
    memory_clean.str.contains('SSD', case=False),
    memory_clean.str.extract(r'(\d+\s*(?:GB|TB))\s*SSD', flags=2)[0],
    np.nan
)

# If the row contains 'HDD', extract the number sequence just before the word 'HDD'
df['hdd'] = np.where(
    memory_clean.str.contains('HDD', case=False),
    memory_clean.str.extract(r'(\d+\s*(?:GB|TB))\s*HDD', flags=2)[0],
    np.nan
)

In [23]:
def get_value_storage(column,df):
    is_tb = [ df[column].str.contains('GB'), df[column].str.contains('TB')]
    sizes = df[column].str.extract(r'(\d+)')[0].astype(float)
    choice_sizes = [sizes,1024*sizes]
    return np.select(is_tb, choice_sizes,default = 0).astype(int)
    

In [24]:
df['ssd capacity'] = get_value_storage('ssd',df)

In [25]:
clean['ssd capacity'] = get_value_storage('ssd',df)

In [26]:
clean['hdd capacity'] = get_value_storage('hdd',df)

In [28]:
clean[ (clean['ssd capacity'] != 0 ) & (clean['hdd capacity']!=0)]

,Company,RAM (GB),CPU_Frequency (GHz),Weight (kg),Price (Euro),Inches,OpSys,ssd capacity,hdd capacity
21,Lenovo,8,2.5,2.50,999.0,15.6,Windows,128,1024
37,Dell,8,1.6,2.80,979.0,17.3,Windows,128,1024
41,Dell,16,2.8,2.65,1499.0,15.6,Windows,256,1024
47,Asus,8,3.0,3.20,1299.0,17.3,Windows,256,1024
58,MSI,16,2.8,2.43,2449.0,17.3,Windows,256,2048
...,...,...,...,...,...,...,...,...,...
1238,MSI,8,2.6,2.30,1169.0,15.6,Windows,128,1024
1245,Asus,16,2.8,2.50,1600.0,15.6,Windows,256,1024
1247,Asus,16,2.6,2.34,2325.0,15.6,Windows,256,1024
1256,Asus,16,2.6,4.00,1900.0,17.3,Windows,128,1024


In [30]:
clean['is_windows'] = (clean['OpSys'] == 'Windows').astype(int)

In [31]:
clean
clean = clean.drop(columns=['OpSys'])

In [279]:
set(df['GPU_Company'])

{'AMD', 'ARM', 'Intel', 'Nvidia'}

In [280]:
set(df['CPU_Company'])

{'AMD', 'Intel', 'Samsung'}

In [33]:
set(df['Company'])

{'Acer',
 'Apple',
 'Asus',
 'Chuwi',
 'Dell',
 'Fujitsu',
 'Google',
 'HP',
 'Huawei',
 'LG',
 'Lenovo',
 'MSI',
 'Mediacom',
 'Microsoft',
 'Razer',
 'Samsung',
 'Toshiba',
 'Vero',
 'Xiaomi'}

In [43]:
set(df['TypeName'])

{'2 in 1 Convertible',
 'Gaming',
 'Netbook',
 'Notebook',
 'Ultrabook',
 'Workstation'}

In [46]:
set(df['ScreenResolution'])

{'1366x768',
 '1440x900',
 '1600x900',
 '1920x1080',
 '2560x1440',
 '4K Ultra HD / Touchscreen 3840x2160',
 '4K Ultra HD 3840x2160',
 'Full HD / Touchscreen 1920x1080',
 'Full HD 1920x1080',
 'IPS Panel 1366x768',
 'IPS Panel 2560x1440',
 'IPS Panel 4K Ultra HD / Touchscreen 3840x2160',
 'IPS Panel 4K Ultra HD 3840x2160',
 'IPS Panel Full HD / Touchscreen 1920x1080',
 'IPS Panel Full HD 1366x768',
 'IPS Panel Full HD 1920x1080',
 'IPS Panel Full HD 1920x1200',
 'IPS Panel Full HD 2160x1440',
 'IPS Panel Full HD 2560x1440',
 'IPS Panel Quad HD+ / Touchscreen 3200x1800',
 'IPS Panel Quad HD+ 2560x1440',
 'IPS Panel Quad HD+ 3200x1800',
 'IPS Panel Retina Display 2304x1440',
 'IPS Panel Retina Display 2560x1600',
 'IPS Panel Retina Display 2736x1824',
 'IPS Panel Retina Display 2880x1800',
 'IPS Panel Touchscreen / 4K Ultra HD 3840x2160',
 'IPS Panel Touchscreen 1366x768',
 'IPS Panel Touchscreen 1920x1200',
 'IPS Panel Touchscreen 2400x1600',
 'IPS Panel Touchscreen 2560x1440',
 'Quad HD

In [39]:
clean['is_touch'] = df['ScreenResolution'].str.contains('Touch|touch').astype(int)

In [41]:
clean['TypeName'] = df['TypeName']

In [44]:
clean['Product'] = df['Product']

In [45]:
clean

,Company,RAM (GB),CPU_Frequency (GHz),Weight (kg),Price (Euro),Inches,ssd capacity,hdd capacity,is_windows,is_touch,TypeName,Product
0,Apple,8,2.3,1.37,1339.69,13.3,128,0,0,0,Ultrabook,MacBook Pro
1,Apple,8,1.8,1.34,898.94,13.3,128,0,0,0,Ultrabook,Macbook Air
2,HP,8,2.5,1.86,575.00,15.6,256,0,1,0,Notebook,250 G6
3,Apple,16,2.7,1.83,2537.45,15.4,512,0,0,0,Ultrabook,MacBook Pro
4,Apple,8,3.1,1.37,1803.60,13.3,256,0,0,0,Ultrabook,MacBook Pro
...,...,...,...,...,...,...,...,...,...,...,...,...
1270,Lenovo,4,2.5,1.80,638.00,14.0,128,0,1,1,2 in 1 Convertible,Yoga 500-14ISK
1271,Lenovo,16,2.5,1.30,1499.00,13.3,512,0,1,1,2 in 1 Convertible,Yoga 900-13ISK
1272,Lenovo,2,1.6,1.50,229.00,14.0,64,0,1,0,Notebook,IdeaPad 100S-14IBR
1273,HP,6,2.5,2.19,764.00,15.6,0,1024,1,0,Notebook,15-AC110nv (i7-6500U/6GB/1TB/Radeon
